# Buby Solidale: pulizia dei dati e preparazione per i Web Solutions Architect
**Versione 3.** Aggiornamento della Versione 2 con risoluzione deterministica delle anomalie nei dati di origine e impostazione di `NON_ASSEGNATO` per i turni scoperti.

**Cosa fa, in 6 passi**
1. Estrae e carica i 7 file grezzi (4 CSV e 3 JSON).
2. Controlla la qualità dei dati (chiavi, date, valori, coerenza tra tabelle, privacy) e registra le anomalie rilevate sui dati grezzi.
3. Pulisce i 4 CSV (date ISO, turni scoperti con `NON_ASSEGNATO`, pseudonimizzazione dei beneficiari con HMAC-SHA256, sanificazione delle anomalie, KPI).
4. Elabora i 3 JSON (maschera i dati personali, estrae kg e urgenza dalle email, classifica l'intento dei messaggi).
5. Verifica il risultato con test automatici (inclusa l'assenza delle anomalie corrette).
6. Crea lo zip con i file puliti, il dizionario dati, il report qualità e il manifest.

**Cosa cambia nella Versione 3 (rispetto alla Versione 2)**

| Area | Versione 2 | Versione 3 |
|---|---|---|
| Turni scoperti | Valore vuoto (`NULL`) di default | `USA_SENTINELLA_VOLONTARIO = True` -> `NON_ASSEGNATO` di default |
| Turni scoperti nel passato (6 casi) | Solo segnalati | Riclassificati in `Annullato` (data precedente a `DATA_RIF`) |
| Doppie assegnazioni volontari (7 casi / 14 turni) | Solo segnalati | Mantenuto il primo turno assegnato (per ID); il secondo turno sovrapposto viene liberato (`NON_ASSEGNATO`, `Scoperto`) |
| Attributi beneficiari variabili (46 beneficiari) | Solo segnalati | Uniformati per persona (`id_beneficiario_anonimo`) usando il profilo della prima richiesta registrata |
| Stato scorte `giacenze` (`G_031` fuori regola) | Solo segnalato | Ricalcolato con regola unica: `CARENZA` se `< soglia_minima_kg`, `RISCHIO_SPRECO` se quota scadenza 48h `>= 20%`, altrimenti `OTTIMALE` |
| Combinazioni uniche quasi-identificatori (26 casi) | Solo segnalate | Documentate nel report qualità (nessuna sovrascrittura: dati sintetici) |

## 1. Configurazione ed estrazione dei file

In [1]:
import os, re, json, zipfile, hashlib, hmac, secrets, shutil
from collections import Counter
import pandas as pd

ZIP_INPUT = "dataset_sintetico_buby_solidale.zip"
DIR_GREZZI = "dati_grezzi"
DIR_PULITI = "dati_puliti_per_wsa"
ZIP_OUTPUT = "dati_puliti_per_wsa.zip"

# Decisione da concordare con i WSA: come rappresentare i turni scoperti.
#   True  -> id_volontario = "NON_ASSEGNATO" (scelta concordata dal team).
#   False -> id_volontario resta vuoto (NULL in SQL).
USA_SENTINELLA_VOLONTARIO = True

FMT_DATA = "%Y-%m-%d"
FMT_DATAORA = "%Y-%m-%dT%H:%M:%S"


def _leggi_sale():
    """Il sale segreto per gli ID dei beneficiari non va mai scritto nel codice né nel repository."""
    sale = os.environ.get("BUBY_SALT")
    if not sale:
        try:
            from google.colab import userdata
            sale = userdata.get("BUBY_SALT")
        except Exception:
            sale = None
    return sale


SALE = _leggi_sale()
if SALE:
    print("Sale letto da BUBY_SALT: gli ID dei beneficiari sono riproducibili tra esecuzioni.")
else:
    SALE = secrets.token_hex(32)
    print("ATTENZIONE: BUBY_SALT non impostato. Uso un sale casuale: gli ID dei beneficiari "
          "cambiano a ogni esecuzione. Per ID stabili imposta BUBY_SALT (variabile d'ambiente o segreto di Colab).")

if not os.path.exists(ZIP_INPUT):
    try:
        from google.colab import files
        print(f"Carica il file {ZIP_INPUT}")
        files.upload()
    except ImportError:
        raise FileNotFoundError(f"Metti {ZIP_INPUT} nella stessa cartella del notebook.")

os.makedirs(DIR_GREZZI, exist_ok=True)
with zipfile.ZipFile(ZIP_INPUT) as z:
    z.extractall(DIR_GREZZI)

FILE_ATTESI = {
    "donazioni.csv", "giacenze_punti_ritiro.csv", "distribuzioni_beneficiari.csv", "turni_volontari.csv",
    "note_operatori_sociali.json", "email_donatori.json", "messaggi_volontari.json",
}
mancanti = FILE_ATTESI - set(os.listdir(DIR_GREZZI))
assert not mancanti, f"File mancanti nello zip: {mancanti}"
print("Estratti i 7 file attesi in", DIR_GREZZI)

ATTENZIONE: BUBY_SALT non impostato. Uso un sale casuale: gli ID dei beneficiari cambiano a ogni esecuzione. Per ID stabili imposta BUBY_SALT (variabile d'ambiente o segreto di Colab).
Estratti i 7 file attesi in dati_grezzi


## 2. Caricamento e controlli di qualità sui dati grezzi
Ogni controllo che trova un problema lo aggiunge alla lista `anomalie`, con la gravità e l'azione presa. Il report finale usa questa lista.

In [2]:
df_donazioni = pd.read_csv(f"{DIR_GREZZI}/donazioni.csv")
df_giacenze = pd.read_csv(f"{DIR_GREZZI}/giacenze_punti_ritiro.csv")
df_distribuzioni = pd.read_csv(f"{DIR_GREZZI}/distribuzioni_beneficiari.csv")
df_turni = pd.read_csv(f"{DIR_GREZZI}/turni_volontari.csv")


def carica_json(nome):
    with open(f"{DIR_GREZZI}/{nome}", encoding="utf-8") as f:
        return json.load(f)


note = carica_json("note_operatori_sociali.json")
email = carica_json("email_donatori.json")
messaggi = carica_json("messaggi_volontari.json")

tabelle = {
    "donazioni": df_donazioni, "giacenze": df_giacenze,
    "distribuzioni": df_distribuzioni, "turni": df_turni,
}
for nome, df in tabelle.items():
    nulli = {c: int(n) for c, n in df.isnull().sum().items() if n > 0}
    print(f"{nome:14s} righe={len(df):4d} colonne={df.shape[1]:2d} nulli={nulli or 'nessuno'}")
print(f"JSON: note={len(note)} email={len(email)} messaggi={len(messaggi)}")

donazioni      righe= 200 colonne=11 nulli=nessuno
giacenze       righe=  35 colonne=10 nulli=nessuno
distribuzioni  righe= 300 colonne=11 nulli=nessuno
turni          righe= 120 colonne= 8 nulli={'id_volontario': 13}
JSON: note=3 email=5 messaggi=4


In [ ]:
anomalie = []


def segnala(gravita, tabella, controllo, n, dettaglio, azione):
    anomalie.append(dict(gravita=gravita, tabella=tabella, controllo=controllo,
                         casi=int(n), dettaglio=dettaglio, azione=azione))
    print(f"[ANOMALIA {gravita}] {tabella}: {controllo} -> {n} casi")


def ok(tabella, controllo):
    print(f"[OK] {tabella}: {controllo}")





# --- Chiavi primarie ---
for nome, df, pk in [("donazioni", df_donazioni, "id_donazione"), ("distribuzioni", df_distribuzioni, "id_distribuzione"),
                     ("turni", df_turni, "id_turno"), ("giacenze", df_giacenze, "id_giacenza")]:
    n = int(df[pk].duplicated().sum())
    segnala("ALTA", nome, f"{pk} duplicato", n, "", "Solo segnalato") if n else ok(nome, f"{pk} univoco")

# --- Chiavi esterne: tutti i punti di ritiro devono esistere in giacenze ---
punti = set(df_giacenze["id_punto_ritiro"])
for nome, df in [("donazioni", df_donazioni), ("distribuzioni", df_distribuzioni), ("turni", df_turni)]:
    orfani = set(df["id_punto_ritiro"]) - punti
    segnala("ALTA", nome, "id_punto_ritiro non presente in giacenze", len(orfani), str(orfani), "Solo segnalato") \
        if orfani else ok(nome, "id_punto_ritiro presente in giacenze")

nomi_punti = df_giacenze.drop_duplicates("id_punto_ritiro").set_index("id_punto_ritiro")["nome_punto_ritiro"]
incoerenti = df_turni[df_turni["nome_punto_ritiro"] != df_turni["id_punto_ritiro"].map(nomi_punti)]
segnala("MEDIA", "turni", "nome_punto_ritiro diverso da giacenze", len(incoerenti), "", "Solo segnalato") \
    if len(incoerenti) else ok("turni", "nome_punto_ritiro coerente con giacenze")

cat_giac = set(df_giacenze["categoria_bene"])
for nome, df, col in [("donazioni", df_donazioni, "categoria_bene"), ("distribuzioni", df_distribuzioni, "categoria_ritirata")]:
    extra = set(df[col]) - cat_giac
    segnala("MEDIA", nome, f"{col} non presente in giacenze", len(extra), str(extra), "Solo segnalato") \
        if extra else ok(nome, f"{col} presente tra le categorie di giacenze")

# --- Date ---
def date_valide(serie, formato):
    return pd.to_datetime(serie, format=formato, errors="coerce")


for nome, df, col, fmt in [("donazioni", df_donazioni, "data_ricezione", FMT_DATA),
                           ("donazioni", df_donazioni, "data_scadenza", FMT_DATA),
                           ("distribuzioni", df_distribuzioni, "data_distribuzione", FMT_DATA),
                           ("turni", df_turni, "data_turno", FMT_DATA),
                           ("giacenze", df_giacenze, "ultimo_aggiornamento", "%Y-%m-%d %H:%M")]:
    n = int(date_valide(df[col], fmt).isna().sum())
    segnala("ALTA", nome, f"{col} non nel formato {fmt}", n, "", "Solo segnalato") if n else ok(nome, f"{col} valida")

n = int((date_valide(df_donazioni["data_scadenza"], FMT_DATA) < date_valide(df_donazioni["data_ricezione"], FMT_DATA)).sum())
segnala("MEDIA", "donazioni", "data_scadenza precedente a data_ricezione", n, "", "Solo segnalato") \
    if n else ok("donazioni", "data_scadenza non precedente a data_ricezione")

# Data di riferimento del dataset: giorno dell'ultimo aggiornamento delle giacenze
DATA_RIF = date_valide(df_giacenze["ultimo_aggiornamento"], "%Y-%m-%d %H:%M").max().normalize()
print("Data di riferimento del dataset:", DATA_RIF.date())

# --- Valori ---
for nome, df, col in [("donazioni", df_donazioni, "quantita_kg"), ("distribuzioni", df_distribuzioni, "kg_distribuiti")]:
    n = int((df[col] <= 0).sum())
    segnala("MEDIA", nome, f"{col} <= 0", n, "", "Solo segnalato") if n else ok(nome, f"{col} sempre positivo")

n = int(((df_distribuzioni["pasti_equivalenti"] - 2 * df_distribuzioni["kg_distribuiti"]).abs() > 1).sum())
segnala("BASSA", "distribuzioni", "pasti_equivalenti lontano da 2 pasti per kg", n, "", "Solo segnalato") \
    if n else ok("distribuzioni", "pasti_equivalenti coerente con circa 2 pasti per kg")

# --- Turni ---
n = int(((df_turni["stato_copertura"] == "Scoperto") != df_turni["id_volontario"].isna()).sum())
segnala("MEDIA", "turni", "stato_copertura incoerente con id_volontario", n, "", "Solo segnalato") \
    if n else ok("turni", "id_volontario vuoto solo nei turni Scoperti")

turni_dt = date_valide(df_turni["data_turno"], FMT_DATA)
n = int(((df_turni["stato_copertura"] == "Scoperto") & (turni_dt < DATA_RIF)).sum())
if n:
    segnala("BASSA", "turni", "turni Scoperti con data precedente alla data di riferimento", n,
            "Un turno passato dovrebbe essere Completato o annullato", "Corretto: stato_copertura impostato a Annullato")

assegnati = df_turni.dropna(subset=["id_volontario"])
doppi = assegnati[assegnati.duplicated(["id_volontario", "data_turno", "fascia_oraria"], keep=False)]
n_doppi = int(assegnati.duplicated(["id_volontario", "data_turno", "fascia_oraria"]).sum())
if n_doppi:
    segnala("MEDIA", "turni", "stesso volontario in due turni nella stessa data e fascia", n_doppi,
            f"{len(doppi)} turni coinvolti, anche in punti di ritiro diversi", "Corretto: mantenuto il primo turno per ID, liberato il secondo (NON_ASSEGNATO / Scoperto)")

# --- Giacenze ---
carenza_ok = ((df_giacenze["quantita_disponibile_kg"] < df_giacenze["soglia_minima_kg"]) == (df_giacenze["stato_scorte"] == "CARENZA")).all()
ok("giacenze", "CARENZA coincide con quantita < soglia minima") if carenza_ok else \
    segnala("MEDIA", "giacenze", "CARENZA non coincide con quantita < soglia", 1, "", "Solo segnalato")

quota = df_giacenze["kg_in_scadenza_48h"] / df_giacenze["quantita_disponibile_kg"]
quota_min_rischio = quota[df_giacenze["stato_scorte"] == "RISCHIO_SPRECO"].min()
n = int(((df_giacenze["stato_scorte"] == "OTTIMALE") & (quota >= quota_min_rischio)).sum())
if n:
    segnala("BASSA", "giacenze", "stato OTTIMALE con quota in scadenza pari o superiore alla più bassa classificata RISCHIO_SPRECO", n,
            f"soglia implicita circa {quota_min_rischio:.0%}; manca una regola unica per stato_scorte", "Corretto: ricalcolato stato_scorte con soglia unica >= 20% in scadenza 48h")

# --- Beneficiari: gli attributi di una persona devono essere stabili tra le sue richieste ---
attr = df_distribuzioni.groupby("id_beneficiario_anonimo")[["fascia_eta_richiedente", "componenti_nucleo_familiare", "quartiere_residenza"]].nunique()
n = int((attr.max(axis=1) > 1).sum())
ripetuti = int((df_distribuzioni["id_beneficiario_anonimo"].value_counts() > 1).sum())
if n:
    segnala("MEDIA", "distribuzioni", "stesso beneficiario con fascia d'età, nucleo o quartiere diversi tra richieste", n,
            f"{n} beneficiari su {ripetuti} con più richieste: gli attributi sembrano generati per riga e non per persona",
            "Corretto: attributi uniformati per singolo beneficiario (prima richiesta cronologica)")

# --- Messaggi: i volontari devono esistere nei turni ---
vol_turni = set(df_turni["id_volontario"].dropna())
n = sum(1 for m in messaggi if m["id_volontario"] not in vol_turni)
segnala("MEDIA", "messaggi_volontari", "id_volontario assente dai turni", n, "", "Solo segnalato") \
    if n else ok("messaggi_volontari", "tutti gli id_volontario esistono nei turni")

# --- Privacy: l'ID pseudonimo è ricostruibile dal nome? ---
col_nome = "campo_raw_da_anonimizzare_nome"
riproducibili = sum(
    "BEN_" + hashlib.sha256(nome.encode("utf-8")).hexdigest()[:10].upper() == pid
    for nome, pid in zip(df_distribuzioni[col_nome], df_distribuzioni["id_beneficiario_anonimo"])
)
if riproducibili:
    segnala("ALTA", "distribuzioni", "id_beneficiario_anonimo ricostruibile con SHA-256 del nome", riproducibili,
            "Chi ha una lista di nomi può ricavare gli ID con un semplice tentativo", "Corretto: ID rigenerati con HMAC e sale segreto")

gruppi = df_distribuzioni.groupby(["quartiere_residenza", "fascia_eta_richiedente", "componenti_nucleo_familiare"]).size()
n = int((gruppi == 1).sum())
if n:
    segnala("BASSA", "distribuzioni", "combinazioni quartiere + fascia d'età + nucleo con una sola riga", n,
            "Rischio teorico su dati sintetici (k-anonymity)", "Documentato: nessuna alterazione necessaria su dati sintetici")

# Donazioni non ancora ritirate: non devono contare come kg recuperati
n = int((df_donazioni["stato"] != "Ritirata e Stoccata").sum())
segnala("INFO", "donazioni", "donazioni non ancora ritirate", n,
        f"{df_donazioni.loc[df_donazioni['stato'] != 'Ritirata e Stoccata', 'quantita_kg'].sum():.1f} kg",
        "Corretto: esclusi dal KPI dei kg recuperati") if n else None

print(f"\nTotale anomalie registrate: {len(anomalie)}")

[OK] donazioni: id_donazione univoco
[OK] distribuzioni: id_distribuzione univoco
[OK] turni: id_turno univoco
[OK] giacenze: id_giacenza univoco
[OK] donazioni: id_punto_ritiro presente in giacenze
[OK] distribuzioni: id_punto_ritiro presente in giacenze
[OK] turni: id_punto_ritiro presente in giacenze
[OK] turni: nome_punto_ritiro coerente con giacenze
[OK] donazioni: categoria_bene presente tra le categorie di giacenze
[OK] distribuzioni: categoria_ritirata presente tra le categorie di giacenze
[OK] donazioni: data_ricezione valida
[OK] donazioni: data_scadenza valida
[OK] distribuzioni: data_distribuzione valida
[OK] turni: data_turno valida
[OK] giacenze: ultimo_aggiornamento valida
[OK] donazioni: data_scadenza non precedente a data_ricezione
Data di riferimento del dataset: 2026-11-12
[OK] donazioni: quantita_kg sempre positivo
[OK] distribuzioni: kg_distribuiti sempre positivo
[OK] distribuzioni: pasti_equivalenti coerente con circa 2 pasti per kg
[OK] turni: id_volontario vuot

## 3. Pulizia dei 4 CSV e risoluzione delle anomalie
- **Date:** convertite con formato esplicito e riscritte in ISO 8601 (`YYYY-MM-DD` e `YYYY-MM-DDTHH:MM:SS`).
- **Turni scoperti e anomalie turni:**
  - Le celle vuote di `id_volontario` vengono valorizzate con `NON_ASSEGNATO` (`USA_SENTINELLA_VOLONTARIO = True`).
  - **Doppie assegnazioni (7 casi / 14 turni):** a parità di `(id_volontario, data_turno, fascia_oraria)` si mantiene il primo turno assegnato (ordinato per `id_turno`) e si libera il conflitto (`id_volontario = 'NON_ASSEGNATO'`, `stato_copertura = 'Scoperto'`).
  - **Turni scoperti nel passato:** i turni `Scoperto` con `data_turno < DATA_RIF` (inclusi eventuali turni liberati per conflitto nel passato) vengono impostati a `Annullato`.
- **Beneficiari:**
  - Il nome reale viene eliminato e l'ID viene rigenerato con HMAC-SHA256 e sale segreto (`BEN_` + 10 caratteri).
  - **Attributi variabili (46 beneficiari):** per ogni `id_beneficiario_anonimo` si uniformano `fascia_eta_richiedente`, `componenti_nucleo_familiare` e `quartiere_residenza` prendendo quelli della prima richiesta cronologica.
- **Giacenze (`stato_scorte`):** ricalcolato con regola deterministica unica (`CARENZA` se `quantita_disponibile_kg < soglia_minima_kg`, `RISCHIO_SPRECO` se `kg_in_scadenza_48h / quantita_disponibile_kg >= 0.20`, altrimenti `OTTIMALE`).

In [4]:
os.makedirs(DIR_PULITI, exist_ok=True)

# 1. Date: conversione esplicita
df_donazioni_clean = df_donazioni.copy()
for col in ["data_ricezione", "data_scadenza"]:
    df_donazioni_clean[col] = pd.to_datetime(df_donazioni_clean[col], format=FMT_DATA, errors="raise")

df_giacenze_clean = df_giacenze.copy()
df_giacenze_clean["ultimo_aggiornamento"] = pd.to_datetime(df_giacenze_clean["ultimo_aggiornamento"], format="%Y-%m-%d %H:%M", errors="raise")

# Regola unica per stato_scorte (risolve l'anomalia G_031 con 34% in scadenza marcato OTTIMALE)
quota_scad = df_giacenze_clean["kg_in_scadenza_48h"] / df_giacenze_clean["quantita_disponibile_kg"]
df_giacenze_clean["stato_scorte"] = "OTTIMALE"
df_giacenze_clean.loc[quota_scad >= 0.20, "stato_scorte"] = "RISCHIO_SPRECO"
df_giacenze_clean.loc[df_giacenze_clean["quantita_disponibile_kg"] < df_giacenze_clean["soglia_minima_kg"], "stato_scorte"] = "CARENZA"

# 2. Turni: risoluzione doppie assegnazioni, turni scoperti nel passato e sentinella NON_ASSEGNATO
df_turni_clean = df_turni.copy()
df_turni_clean["data_turno"] = pd.to_datetime(df_turni_clean["data_turno"], format=FMT_DATA, errors="raise")

# Risoluzione doppie assegnazioni: ordiniamo per id_turno e liberiamo il secondo slot sovrapposto
df_turni_clean = df_turni_clean.sort_values("id_turno").reset_index(drop=True)
maschera_assegnati = df_turni_clean["id_volontario"].notna()
maschera_doppi = maschera_assegnati & df_turni_clean.duplicated(["id_volontario", "data_turno", "fascia_oraria"], keep="first")
df_turni_clean.loc[maschera_doppi, "id_volontario"] = pd.NA
df_turni_clean.loc[maschera_doppi, "stato_copertura"] = "Scoperto"

# Turni Scoperti con data precedente a DATA_RIF -> Annullato
maschera_passati_scoperti = (df_turni_clean["stato_copertura"] == "Scoperto") & (df_turni_clean["data_turno"] < DATA_RIF)
df_turni_clean.loc[maschera_passati_scoperti, "stato_copertura"] = "Annullato"

# Valorizzazione turni senza volontario con NON_ASSEGNATO
if USA_SENTINELLA_VOLONTARIO:
    df_turni_clean["id_volontario"] = df_turni_clean["id_volontario"].fillna("NON_ASSEGNATO")

# 3. Distribuzioni: pseudonimizzazione HMAC-SHA256 e uniformazione attributi beneficiari
df_distribuzioni_clean = df_distribuzioni.copy()
df_distribuzioni_clean["data_distribuzione"] = pd.to_datetime(df_distribuzioni_clean["data_distribuzione"], format=FMT_DATA, errors="raise")


def pseudonimo(nome_raw):
    """HMAC-SHA256 con sale segreto: senza il sale non si risale al nome neppure con una lista di nomi."""
    h = hmac.new(SALE.encode("utf-8"), nome_raw.encode("utf-8"), hashlib.sha256).hexdigest()
    return "BEN_" + h[:10].upper()


# Teniamo in memoria (mai su disco) i nomi originali: servono per il dizionario di mascheramento e per i test finali
_nomi_raw = df_distribuzioni[col_nome].tolist()
_id_vecchi = df_distribuzioni["id_beneficiario_anonimo"].tolist()
NOMI_BENEFICIARI = sorted({re.sub(r"\s*#\d+$", "", n).strip() for n in _nomi_raw})

nuovi_id = df_distribuzioni[col_nome].map(pseudonimo)
assert nuovi_id.nunique() == pd.Series(_id_vecchi).nunique(), "La corrispondenza persona-ID deve restare 1 a 1"
df_distribuzioni_clean["id_beneficiario_anonimo"] = nuovi_id
df_distribuzioni_clean = df_distribuzioni_clean.drop(columns=[col_nome])

# Uniformazione attributi anagrafici per beneficiario (prendiamo il profilo della prima richiesta cronologica)
colonne_profilo = ["fascia_eta_richiedente", "componenti_nucleo_familiare", "quartiere_residenza"]
profilo_canonico = (
    df_distribuzioni_clean
    .sort_values(["data_distribuzione", "id_distribuzione"])
    .drop_duplicates("id_beneficiario_anonimo", keep="first")
    .set_index("id_beneficiario_anonimo")[colonne_profilo]
)
for col in colonne_profilo:
    df_distribuzioni_clean[col] = df_distribuzioni_clean["id_beneficiario_anonimo"].map(profilo_canonico[col])

print(f"Turni: risolti {int(maschera_doppi.sum())} conflitti doppi, {int(maschera_passati_scoperti.sum())} turni passati impostati ad Annullato")
print("Distribuzioni:", df_distribuzioni.shape, "->", df_distribuzioni_clean.shape, "| ID beneficiari:", nuovi_id.nunique())

Turni: risolti 7 conflitti doppi, 9 turni passati impostati ad Annullato
Distribuzioni: (300, 11) -> (300, 10) | ID beneficiari: 251


In [5]:
# KPI. I kg recuperati contano solo le donazioni effettivamente ritirate
ritirate = df_donazioni_clean[df_donazioni_clean["stato"] == "Ritirata e Stoccata"]
kpi = {
    "kg_recuperati": round(float(ritirate["quantita_kg"].sum()), 1),
    "kg_in_attesa_di_ritiro": round(float(df_donazioni_clean.loc[df_donazioni_clean["stato"] != "Ritirata e Stoccata", "quantita_kg"].sum()), 1),
    "kg_distribuiti": round(float(df_distribuzioni_clean["kg_distribuiti"].sum()), 1),
    "pasti_equivalenti": int(df_distribuzioni_clean["pasti_equivalenti"].sum()),
    "beneficiari_unici": int(df_distribuzioni_clean["id_beneficiario_anonimo"].nunique()),
    "turni_totali": int(len(df_turni_clean)),
    "turni_scoperti_futuri": int((df_turni_clean["stato_copertura"] == "Scoperto").sum()),
    "turni_annullati_passati": int((df_turni_clean["stato_copertura"] == "Annullato").sum()),
}
kpi["turni_scoperti_pct"] = round(100 * kpi["turni_scoperti_futuri"] / kpi["turni_totali"], 1)
for k, v in kpi.items():
    print(f"{k:25s} {v}")

kg_recuperati             26105.3
kg_in_attesa_di_ritiro    243.1
kg_distribuiti            2467.3
pasti_equivalenti         4929
beneficiari_unici         251
turni_totali              120
turni_scoperti_futuri     11
turni_annullati_passati   9
turni_scoperti_pct        9.2


In [6]:
def esporta_csv(df, nome, colonne_data=(), colonne_datetime=()):
    out = df.copy()
    for c in colonne_data:
        out[c] = out[c].dt.strftime(FMT_DATA)
    for c in colonne_datetime:
        out[c] = out[c].dt.strftime(FMT_DATAORA)
    out.to_csv(f"{DIR_PULITI}/{nome}", index=False, na_rep="")


esporta_csv(df_donazioni_clean, "donazioni_clean.csv", colonne_data=["data_ricezione", "data_scadenza"])
esporta_csv(df_giacenze_clean, "giacenze_clean.csv", colonne_datetime=["ultimo_aggiornamento"])
esporta_csv(df_distribuzioni_clean, "distribuzioni_clean.csv", colonne_data=["data_distribuzione"])
esporta_csv(df_turni_clean, "turni_clean.csv", colonne_data=["data_turno"])
print("Salvati i 4 CSV in", DIR_PULITI)

Salvati i 4 CSV in dati_puliti_per_wsa


## 4. Pulizia dei 3 JSON
Ordine delle operazioni: prima si **maschera** il testo, poi l'estrazione NLP lavora sul testo già mascherato, così nessun dato personale passa per i passi successivi.

Le regole di mascheramento sono: codice fiscale, email, telefono (mobile e fisso, con o senza +39), nome dopo un titolo (sig., signora, dott.ssa e simili), nome che compare in un dizionario dei beneficiari noti, nome dopo una formula di saluto finale. Un nome senza titolo e fuori dal dizionario non può essere riconosciuto da una regola: per quel caso servirebbe un modello di riconoscimento delle entità (NER).

In [7]:
MAI, MIN = "A-ZÀ-ÖØ-Þ", "a-zß-öø-ÿ"
PAROLA_NOME = rf"(?:[{MAI}][{MIN}]?['’])?[{MAI}][{MIN}]+(?:-[{MAI}][{MIN}]+)*"
NOME_COMPLETO = PAROLA_NOME + rf"(?:\s+{PAROLA_NOME}){{0,3}}"
# Il flag (?i) vale solo per il titolo: i nomi devono restare case-sensitive
TITOLI = (r"(?i:\b(?:sig\.\s?ra|sig\.\s?na|sig\.|signora|signorina|signor|sig|dott\.\s?ssa|dott\.|dottoressa|dottor|"
          r"prof\.\s?ssa|prof\.|professoressa|professor|avv\.|ing\.)\s+)")

RE_TITOLO_NOME = re.compile("(" + TITOLI + ")" + NOME_COMPLETO)
RE_CF = re.compile(r"\b[A-Za-z]{6}\d{2}[A-Za-z]\d{2}[A-Za-z]\d{3}[A-Za-z]\b")
RE_EMAIL = re.compile(r"[\w.+\-]+@[\w\-]+(?:\.[\w\-]+)+")
RE_TEL = re.compile(r"(?<!\w)(?:(?:\+|00)39[\s.\-]?)?(?:3\d{2}[\s.\-]?\d{6,7}|0\d{1,3}[\s.\-]?\d{5,8})(?!\w)")
RE_FIRMA = re.compile(r"((?i:grazie|cordiali saluti|distinti saluti|saluti|a presto|buon lavoro)[\s.,!]*)(" + PAROLA_NOME + r")(\.?\s*)$")
RE_NOMI_NOTI = re.compile(r"\b(?:" + "|".join(re.escape(n) for n in sorted(NOMI_BENEFICIARI, key=len, reverse=True)) + r")\b", re.IGNORECASE)


def anonimizza_testo(testo, contatori=None):
    c = contatori if contatori is not None else Counter()

    def passo(rx, rep, chiave, t):
        t, n = rx.subn(rep, t)
        c[chiave] += n
        return t

    t = testo
    t = passo(RE_CF, "[CF_OSCURATO]", "codici_fiscali", t)
    t = passo(RE_EMAIL, "[EMAIL_OSCURATA]", "email", t)
    t = passo(RE_TEL, "[TEL_OSCURATO]", "telefoni", t)
    t = passo(RE_TITOLO_NOME, r"\1[NOME_OSCURATO]", "nomi", t)
    t = passo(RE_NOMI_NOTI, "[NOME_OSCURATO]", "nomi", t)
    t = passo(RE_FIRMA, r"\1[NOME_OSCURATO]\3", "nomi", t)
    return t

In [8]:
RE_KG = re.compile(r"(\d+(?:[.,]\d+)?)\s*(?:kg|chili|chilogrammi)\b", re.IGNORECASE)

REGOLE_URGENZA = [
    ("urgente", re.compile(r"\burgent\w*", re.IGNORECASE)),
    ("rischio_spreco", re.compile(r"\bmacero\b|\bbuttare\b|\bsmaltire\b|\bspreco\b", re.IGNORECASE)),
    ("scadenza_ravvicinata", re.compile(r"\bin scadenza\b|\b(?:24|48|72)\s*ore\b", re.IGNORECASE)),
    ("ritiro_in_giornata", re.compile(r"\bstasera\b|\boggi\b|\bentro (?:oggi|domani|stasera|le \d{1,2}[:.]\d{2})", re.IGNORECASE)),
]

REGOLE_INTENTO = [  # l'ordine conta: la prima regola che corrisponde vince
    ("DISDETTA_TURNO", re.compile(r"\bdisdir\w*|\bdisdic\w*|\bannull\w*|\bsostituir\w*|\bnon (?:posso|riesco|ce la faccio|potr\w*)\b", re.IGNORECASE)),
    ("SEGNALAZIONE_RITARDO", re.compile(r"\britard\w*|\barrivo tardi\b", re.IGNORECASE)),
    ("DISPONIBILITA_TURNO", re.compile(r"\bdisponibil\w*|\bsono liber[oa]\b|\bci sono\b|\bconfermo\b|\bposso (?:fare|coprire|venire)\b", re.IGNORECASE)),
]


def estrai_kg(testo):
    """Ritorna la lista dei kg citati. Limite noto: se l'email cita anche un totale, viene contato due volte."""
    valori = []
    for m in RE_KG.findall(testo):
        v = float(m.replace(",", "."))
        valori.append(int(v) if v.is_integer() else v)
    return valori


def valuta_urgenza(testo):
    motivi = [nome for nome, rx in REGOLE_URGENZA if rx.search(testo)]
    return ("ALTA" if motivi else "NORMALE"), motivi


def classifica_intento(testo):
    for nome, rx in REGOLE_INTENTO:
        if rx.search(testo):
            return nome
    return "ALTRO"

### Test automatici sulle funzioni
Vengono eseguiti prima di toccare i dati: se uno fallisce, il notebook si ferma.

In [9]:
# 1) Mascheramento su testi inventati, indipendenti dal dataset
casi_privacy = {
    "Il sig. Luca D'Angelo (cell. +39 347 1234567, luca@mail.it) CF RSSMRA80A01L219K":
        "Il sig. [NOME_OSCURATO] (cell. [TEL_OSCURATO], [EMAIL_OSCURATA]) CF [CF_OSCURATO]",
    "La dott.ssa Anna Maria De Luca ha chiamato lo 011 1234567.":
        "La dott.ssa [NOME_OSCURATO] ha chiamato lo [TEL_OSCURATO].",
    "La signora ha segnalato un problema alle 16:30 del 2026-11-12.":  # nessun dato personale: il testo non deve cambiare
        "La signora ha segnalato un problema alle 16:30 del 2026-11-12.",
    "Fatemi sapere, grazie. Marco.":
        "Fatemi sapere, grazie. [NOME_OSCURATO].",
}
for ingresso, atteso in casi_privacy.items():
    ottenuto = anonimizza_testo(ingresso)
    assert ottenuto == atteso, f"Mascheramento errato:\n  in : {ingresso}\n  out: {ottenuto}\n  att: {atteso}"

# 2) NLP su testi inventati
assert estrai_kg("Abbiamo 12,5 kg di pane e 30 kg di pasta") == [12.5, 30]
assert valuta_urgenza("Ritiro disponibile domani mattina, scadenza tra 4 mesi")[0] == "NORMALE"
assert valuta_urgenza("Prodotti in scadenza tra 48 ore")[0] == "ALTA"
assert classifica_intento("Non posso venire domani") == "DISDETTA_TURNO"
assert classifica_intento("Sarò in ritardo di 10 minuti") == "SEGNALAZIONE_RITARDO"
assert classifica_intento("Sono libero giovedì, ci sono") == "DISPONIBILITA_TURNO"
assert classifica_intento("Ciao a tutti, buona giornata") == "ALTRO"
print("Test sulle funzioni: tutti superati")

Test sulle funzioni: tutti superati


In [10]:
contatori_pii = Counter()

# Note degli operatori: solo mascheramento
note_clean = []
for n in note:
    r = dict(n)
    r["testo_nota"] = anonimizza_testo(r["testo_nota"], contatori_pii)
    note_clean.append(r)

# Email: mascheramento, poi estrazione NLP sul testo mascherato
email_clean = []
for e in email:
    r = dict(e)
    r["oggetto"] = anonimizza_testo(r["oggetto"], contatori_pii)
    r["corpo_testo"] = anonimizza_testo(r["corpo_testo"], contatori_pii)
    kg = estrai_kg(r["oggetto"] + " " + r["corpo_testo"])
    urgenza, motivi = valuta_urgenza(r["oggetto"] + " " + r["corpo_testo"])
    somma = round(sum(kg), 1)
    r["nlp_extracted"] = {
        "kg_rilevati": int(somma) if float(somma).is_integer() else somma,
        "kg_dettaglio": kg,
        "urgenza": urgenza,
        "motivi_urgenza": motivi,
    }
    email_clean.append(r)

# Messaggi dei volontari: mascheramento, poi classificazione dell'intento
messaggi_clean = []
for m in messaggi:
    r = dict(m)
    r["testo"] = anonimizza_testo(r["testo"], contatori_pii)
    r["nlp_intent"] = classifica_intento(r["testo"])
    messaggi_clean.append(r)

print("Dati personali oscurati:", dict(contatori_pii))
print()
for e in email_clean:
    print(e["id_email"], e["nlp_extracted"])
for m in messaggi_clean:
    print(m["id_msg"], m["nlp_intent"])

Dati personali oscurati: {'codici_fiscali': 1, 'email': 0, 'telefoni': 1, 'nomi': 3}

EML_001 {'kg_rilevati': 45, 'kg_dettaglio': [45], 'urgenza': 'ALTA', 'motivi_urgenza': ['urgente', 'scadenza_ravvicinata', 'ritiro_in_giornata']}
EML_002 {'kg_rilevati': 22, 'kg_dettaglio': [22], 'urgenza': 'ALTA', 'motivi_urgenza': ['ritiro_in_giornata']}
EML_003 {'kg_rilevati': 100, 'kg_dettaglio': [80, 20], 'urgenza': 'NORMALE', 'motivi_urgenza': []}
EML_004 {'kg_rilevati': 240, 'kg_dettaglio': [180, 60], 'urgenza': 'ALTA', 'motivi_urgenza': ['urgente', 'rischio_spreco', 'ritiro_in_giornata']}
EML_005 {'kg_rilevati': 28, 'kg_dettaglio': [28], 'urgenza': 'NORMALE', 'motivi_urgenza': []}
MSG_001 DISPONIBILITA_TURNO
MSG_002 DISDETTA_TURNO
MSG_003 DISPONIBILITA_TURNO
MSG_004 SEGNALAZIONE_RITARDO


In [11]:
# 3) Test di regressione sul dataset consegnato (valori verificati a mano leggendo i testi)
atteso_email = {"EML_001": (45, "ALTA"), "EML_002": (22, "ALTA"), "EML_003": (100, "NORMALE"),
                "EML_004": (240, "ALTA"), "EML_005": (28, "NORMALE")}
for e in email_clean:
    if e["id_email"] in atteso_email:
        kg_att, urg_att = atteso_email[e["id_email"]]
        assert e["nlp_extracted"]["kg_rilevati"] == kg_att and e["nlp_extracted"]["urgenza"] == urg_att, e["id_email"]

atteso_intenti = {"MSG_001": "DISPONIBILITA_TURNO", "MSG_002": "DISDETTA_TURNO",
                  "MSG_003": "DISPONIBILITA_TURNO", "MSG_004": "SEGNALAZIONE_RITARDO"}
for m in messaggi_clean:
    if m["id_msg"] in atteso_intenti:
        assert m["nlp_intent"] == atteso_intenti[m["id_msg"]], m["id_msg"]
print("Test di regressione sul dataset: superati")

Test di regressione sul dataset: superati


In [12]:
def salva_json(dati, nome):
    with open(f"{DIR_PULITI}/{nome}", "w", encoding="utf-8") as f:
        json.dump(dati, f, ensure_ascii=False, indent=2)


salva_json(note_clean, "note_operatori_clean.json")
salva_json(email_clean, "email_donatori_clean.json")
salva_json(messaggi_clean, "messaggi_volontari_clean.json")
print("Salvati i 3 JSON in", DIR_PULITI)

Salvati i 3 JSON in dati_puliti_per_wsa


## 5. Verifica finale sui file puliti
I controlli rileggono i file dal disco, cioè quello che riceveranno i WSA.

In [13]:
atteso_puliti = {"donazioni_clean.csv", "giacenze_clean.csv", "distribuzioni_clean.csv", "turni_clean.csv",
                 "note_operatori_clean.json", "email_donatori_clean.json", "messaggi_volontari_clean.json"}
assert atteso_puliti <= set(os.listdir(DIR_PULITI))

p_don = pd.read_csv(f"{DIR_PULITI}/donazioni_clean.csv")
p_gia = pd.read_csv(f"{DIR_PULITI}/giacenze_clean.csv")
p_dis = pd.read_csv(f"{DIR_PULITI}/distribuzioni_clean.csv")
p_tur = pd.read_csv(f"{DIR_PULITI}/turni_clean.csv")

# Righe invariate
assert (len(p_don), len(p_gia), len(p_dis), len(p_tur)) == (len(df_donazioni), len(df_giacenze), len(df_distribuzioni), len(df_turni))

# Colonna con i nomi reali assente
assert col_nome not in p_dis.columns

# Formati ISO
for df, col, rx in [(p_don, "data_ricezione", r"\d{4}-\d{2}-\d{2}"), (p_don, "data_scadenza", r"\d{4}-\d{2}-\d{2}"),
                    (p_dis, "data_distribuzione", r"\d{4}-\d{2}-\d{2}"), (p_tur, "data_turno", r"\d{4}-\d{2}-\d{2}"),
                    (p_gia, "ultimo_aggiornamento", r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}")]:
    assert df[col].astype(str).str.fullmatch(rx).all(), f"Formato data non ISO in {col}"

# Nessun nullo: con USA_SENTINELLA_VOLONTARIO=True anche turni_clean.csv ha 0 nulli
for nome, df in [("donazioni", p_don), ("giacenze", p_gia), ("distribuzioni", p_dis)]:
    assert df.isnull().sum().sum() == 0, f"Nulli inattesi in {nome}"
senza_vol = int((p_tur["stato_copertura"].isin(["Scoperto", "Annullato"])).sum())
if USA_SENTINELLA_VOLONTARIO:
    assert p_tur.isnull().sum().sum() == 0 and (p_tur["id_volontario"] == "NON_ASSEGNATO").sum() == senza_vol
else:
    assert p_tur.isnull().sum().sum() == senza_vol and p_tur["id_volontario"].isna().sum() == senza_vol

# Verifica risoluzione anomalie nei file puliti
vol_attivi = p_tur[p_tur["id_volontario"] != "NON_ASSEGNATO"].dropna(subset=["id_volontario"])
assert vol_attivi.duplicated(["id_volontario", "data_turno", "fascia_oraria"]).sum() == 0, "Doppie assegnazioni ancora presenti"
assert ((p_tur["stato_copertura"] == "Scoperto") & (pd.to_datetime(p_tur["data_turno"]) < DATA_RIF)).sum() == 0, "Turni scoperti nel passato ancora presenti"

attr_clean = p_dis.groupby("id_beneficiario_anonimo")[["fascia_eta_richiedente", "componenti_nucleo_familiare", "quartiere_residenza"]].nunique()
assert (attr_clean.max(axis=1) > 1).sum() == 0, "Attributi beneficiari ancora incoerenti"

quota_clean = p_gia["kg_in_scadenza_48h"] / p_gia["quantita_disponibile_kg"]
assert ((p_gia["stato_scorte"] == "OTTIMALE") & (quota_clean >= 0.20)).sum() == 0, "Giacenza OTTIMALE con >= 20% in scadenza"

# Integrità referenziale sui file puliti
assert set(p_don["id_punto_ritiro"]) | set(p_dis["id_punto_ritiro"]) | set(p_tur["id_punto_ritiro"]) <= set(p_gia["id_punto_ritiro"])

# Privacy 1: gli ID non sono piu' ricostruibili con SHA-256 del nome
ricostruibili = sum("BEN_" + hashlib.sha256(n.encode("utf-8")).hexdigest()[:10].upper() in set(p_dis["id_beneficiario_anonimo"]) for n in _nomi_raw)
assert ricostruibili == 0, "Gli ID dei beneficiari sono ancora ricostruibili"

# Privacy 2: nei file puliti non compare nessun nome reale, codice fiscale, telefono o email personale
testo_csv = " ".join(open(f"{DIR_PULITI}/{f}", encoding="utf-8").read() for f in atteso_puliti if f.endswith(".csv"))
testo_json = json.dumps([note_clean, [{k: v for k, v in e.items() if k != "mittente"} for e in email_clean], messaggi_clean], ensure_ascii=False)
for nome in NOMI_BENEFICIARI:
    assert nome.lower() not in (testo_csv + testo_json).lower(), "Un nome reale è rimasto nei file puliti"
assert not RE_CF.search(testo_csv + testo_json), "Codice fiscale rimasto nei file puliti"
assert not RE_TEL.search(testo_json), "Telefono rimasto nei JSON"
assert not RE_EMAIL.search(testo_json), "Email personale rimasta nei JSON"

# JSON: stesso numero di record e nuovi campi presenti
assert (len(note_clean), len(email_clean), len(messaggi_clean)) == (len(note), len(email), len(messaggi))
assert all("nlp_extracted" in e for e in email_clean) and all("nlp_intent" in m for m in messaggi_clean)
print("Verifica finale: tutti i controlli (inclusa la risoluzione delle anomalie) superati!")

Verifica finale: tutti i controlli (inclusa la risoluzione delle anomalie) superati!


## 6. Documentazione di consegna: dizionario dati, report qualità, manifest e zip

In [14]:
DIZIONARIO = {
    "donazioni_clean.csv": ("Una riga per ogni donazione ricevuta.", {
        "id_donazione": "Chiave primaria della donazione",
        "id_donatore": "Identificativo dell'organizzazione donatrice",
        "nome_donatore": "Denominazione dell'organizzazione donatrice",
        "tipologia_donatore": "Tipo di donatore",
        "id_punto_ritiro": "Punto di ritiro. Chiave esterna verso giacenze_clean.csv",
        "categoria_bene": "Categoria merceologica del bene donato",
        "deperibilita": "Livello di deperibilità del bene",
        "quantita_kg": "Peso della donazione in kg",
        "data_ricezione": "Data di ricezione (YYYY-MM-DD)",
        "data_scadenza": "Data di scadenza del bene (YYYY-MM-DD)",
        "stato": "Stato della donazione. Solo 'Ritirata e Stoccata' conta come kg recuperati",
    }),
    "giacenze_clean.csv": ("Una riga per ogni coppia punto di ritiro e categoria di bene: stato delle scorte.", {
        "id_giacenza": "Chiave primaria",
        "id_punto_ritiro": "Punto di ritiro",
        "nome_punto_ritiro": "Nome del punto di ritiro",
        "quartiere": "Quartiere del punto di ritiro",
        "categoria_bene": "Categoria merceologica",
        "quantita_disponibile_kg": "Scorta disponibile in kg",
        "soglia_minima_kg": "Soglia minima di scorta in kg",
        "kg_in_scadenza_48h": "Kg che scadono entro 48 ore",
        "stato_scorte": "CARENZA se scorta < soglia minima; RISCHIO_SPRECO se quota in scadenza 48h >= 20%; altrimenti OTTIMALE",
        "ultimo_aggiornamento": "Data e ora dell'ultimo aggiornamento (YYYY-MM-DDTHH:MM:SS)",
    }),
    "distribuzioni_clean.csv": ("Una riga per ogni distribuzione a un beneficiario. Nessun nome reale.", {
        "id_distribuzione": "Chiave primaria",
        "id_beneficiario_anonimo": "ID pseudonimo del beneficiario (BEN_ + 10 caratteri). Non ricostruibile dal nome",
        "fascia_eta_richiedente": "Fascia d'età del richiedente",
        "componenti_nucleo_familiare": "Numero di componenti del nucleo familiare",
        "quartiere_residenza": "Quartiere di residenza",
        "id_punto_ritiro": "Punto di ritiro. Chiave esterna verso giacenze_clean.csv",
        "categoria_ritirata": "Categoria di bene ritirata",
        "kg_distribuiti": "Peso distribuito in kg",
        "pasti_equivalenti": "Pasti equivalenti stimati dal peso (circa 2 per kg nei dati)",
        "data_distribuzione": "Data della distribuzione (YYYY-MM-DD)",
    }),
    "turni_clean.csv": ("Una riga per ogni turno di volontariato.", {
        "id_turno": "Chiave primaria",
        "id_punto_ritiro": "Punto di ritiro. Chiave esterna verso giacenze_clean.csv",
        "nome_punto_ritiro": "Nome del punto di ritiro",
        "data_turno": "Data del turno (YYYY-MM-DD)",
        "fascia_oraria": "Fascia oraria del turno",
        "mansione": "Mansione richiesta",
        "id_volontario": ("Volontario assegnato. Vuoto (NULL) nei turni Scoperti"
                          if not USA_SENTINELLA_VOLONTARIO else "Volontario assegnato. 'NON_ASSEGNATO' nei turni Scoperti"),
        "stato_copertura": "Completato, Coperto, Scoperto (futuro) o Annullato (passato non coperto)",
    }),
}
DIZIONARIO_JSON = {
    "note_operatori_clean.json": "Note degli operatori sociali con nomi, telefoni e codici fiscali sostituiti da segnaposto "
                                 "([NOME_OSCURATO], [TEL_OSCURATO], [CF_OSCURATO], [EMAIL_OSCURATA]).",
    "email_donatori_clean.json": "Email dei donatori. Campo aggiunto nlp_extracted con: kg_rilevati (somma dei kg citati), "
                                 "kg_dettaglio (i singoli valori), urgenza (ALTA o NORMALE), motivi_urgenza (regole scattate). "
                                 "Estrazione basata su regole, non su un modello.",
    "messaggi_volontari_clean.json": "Messaggi dei volontari. Campo aggiunto nlp_intent: DISPONIBILITA_TURNO, DISDETTA_TURNO, "
                                     "SEGNALAZIONE_RITARDO oppure ALTRO. Classificazione basata su regole.",
}


def tipo(serie):
    if pd.api.types.is_integer_dtype(serie):
        return "intero"
    if pd.api.types.is_float_dtype(serie):
        return "decimale"
    return "testo"


righe = ["# Dizionario dati - Buby Solidale (file puliti)", "",
         "Le descrizioni sono una proposta di Michele e Christian da validare con i WSA, insieme allo schema delle API.", ""]
for nome_file, (descr, colonne) in DIZIONARIO.items():
    df = pd.read_csv(f"{DIR_PULITI}/{nome_file}")
    assert set(colonne) == set(df.columns), f"Il dizionario di {nome_file} non coincide con le colonne"
    righe += [f"## {nome_file}", descr, "", "| Colonna | Tipo | Nullable | Valori ammessi | Descrizione |", "|---|---|---|---|---|"]
    for c in df.columns:
        valori = sorted(df[c].dropna().unique().tolist()) if df[c].nunique() <= 8 and tipo(df[c]) == "testo" else []
        righe.append(f"| {c} | {tipo(df[c])} | {'sì' if df[c].isnull().any() else 'no'} | {', '.join(map(str, valori))} | {colonne[c]} |")
    righe.append("")
for nome_file, descr in DIZIONARIO_JSON.items():
    righe += [f"## {nome_file}", descr, ""]
with open(f"{DIR_PULITI}/data_dictionary.md", "w", encoding="utf-8") as f:
    f.write("\n".join(righe))

In [15]:
ordine = {"ALTA": 0, "MEDIA": 1, "BASSA": 2, "INFO": 3}
righe = ["# Report qualità dati - Buby Solidale", "",
         f"Data di riferimento del dataset: {DATA_RIF.date()}", "",
         "## KPI", "", "| Indicatore | Valore |", "|---|---|"]
righe += [f"| {k} | {v} |" for k, v in kpi.items()]
righe += ["", "## Dati personali oscurati nei JSON", "", "| Tipo | Occorrenze |", "|---|---|"]
righe += [f"| {k} | {v} |" for k, v in sorted(contatori_pii.items())]
righe += ["", "## Anomalie", "", "| Gravità | Tabella | Controllo | Casi | Dettaglio | Azione |", "|---|---|---|---|---|---|"]
for a in sorted(anomalie, key=lambda x: ordine[x["gravita"]]):
    righe.append(f"| {a['gravita']} | {a['tabella']} | {a['controllo']} | {a['casi']} | {a['dettaglio']} | {a['azione']} |")
righe += ["", "## Parametri della pulizia", "",
          f"- Turni scoperti: {'NON_ASSEGNATO' if USA_SENTINELLA_VOLONTARIO else 'valore vuoto (NULL)'}",
          "- ID beneficiari: HMAC-SHA256 con sale segreto, 10 caratteri esadecimali",
          f"- Sale da variabile BUBY_SALT: {'sì' if os.environ.get('BUBY_SALT') else 'no, sale casuale per questa esecuzione'}"]
with open(f"{DIR_PULITI}/report_qualita.md", "w", encoding="utf-8") as f:
    f.write("\n".join(righe))

In [16]:
manifest = []
for nome_file in sorted(os.listdir(DIR_PULITI)):
    if nome_file == "manifest.json":
        continue
    percorso = f"{DIR_PULITI}/{nome_file}"
    contenuto = open(percorso, "rb").read()
    voce = {"file": nome_file, "byte": len(contenuto), "sha256": hashlib.sha256(contenuto).hexdigest()}
    if nome_file.endswith(".csv"):
        voce["righe"] = int(len(pd.read_csv(percorso)))
    elif nome_file.endswith(".json"):
        voce["record"] = len(json.loads(contenuto))
    manifest.append(voce)
with open(f"{DIR_PULITI}/manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

shutil.make_archive("dati_puliti_per_wsa", "zip", root_dir=".", base_dir=DIR_PULITI)
tab = pd.DataFrame(manifest)
for c in ("righe", "record"):
    tab[c] = tab[c].astype("Int64")
print(tab.to_string(index=False))
print(f"\nCreato {ZIP_OUTPUT}")

# I nomi originali servivano solo in memoria: li eliminiamo
del _nomi_raw, _id_vecchi

try:
    from google.colab import files
    files.download(ZIP_OUTPUT)
except ImportError:
    pass

                         file  byte                                                           sha256  righe  record
           data_dictionary.md  6087 d8d1fc676587c966fc93de3358d3222aeea295cf4a068b1088ddceeb7906a012   <NA>    <NA>
      distribuzioni_clean.csv 27967 c4c8ed49b2d5f50b11a7c59d2311c12e3ad6adde809c5a9436fe83ebd2878c44    300    <NA>
          donazioni_clean.csv 28467 2bc7d724f20c388ecec7de82679563307e5ffec324e006526c32ba5f6da51174    200    <NA>
    email_donatori_clean.json  3006 3771b76699549f855c53025635b1487923cabffc9378a3837d7ab4fdb3f63b73   <NA>       5
           giacenze_clean.csv  4142 12b559e13bc245427cb7f4893607bd8a07224fabaaee90f7ac3163a7ea39b150     35    <NA>
messaggi_volontari_clean.json  1247 68b565be129985af34c873ad3688340c80eefabbeb2e426e217991c2dedaf032   <NA>       4
    note_operatori_clean.json  1181 ad9fc498ca622829813e65119ea4bb270cfca8abe94a75ecfa1d71ec2aeb9780   <NA>       3
            report_qualita.md  2372 4a069e0bf6be5cb57ef1e0b92d1bed43fb23

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>